# Notebook 02b — MS 4-Band Stack and Tile

This notebook:
1. Builds a 4-band MS stack per SA: **MS Red + MS Green + NIR + Red-edge**
2. Tiles the MS stack to 512x512 with 256px stride
3. Moves MS tiles into the existing train/val/test split folders

Label tiles are already split from Stage 3 — no relabelling needed.

**Band order in stack:**

| Band | Source file keyword |
|------|--------------------|
| 1 | `mosaic_red` |
| 2 | `mosaic_green` |
| 3 | `mosaic_nir` |
| 4 | `mosaic_red edge`  |

```
Tiles/
  MS/images/       <- 4-band MS tiles
  labels/           <-  label tiles

```

## 0. Imports and paths

In [1]:
from pathlib import Path
import numpy as np
import rasterio
from rasterio.windows import Window
from rasterio.enums import Resampling
import rasterio.warp
import pandas as pd
import shutil

BASE_DIR   = Path(r"E:\THESIS_COLLINS HLORDZIE\02_PROCESSED")
SA_ROOT    = BASE_DIR
LABEL_DIR  = BASE_DIR / "Labels"
TILES_DIR  = BASE_DIR / "Tiles"

MS_IMG_DIR = TILES_DIR / "ms" / "images"
LBL_DIR    = TILES_DIR / "labels"

MS_IMG_DIR.mkdir(parents=True, exist_ok=True)

TILE_SIZE  = 512
STRIDE     = 256
BACKGROUND = 255
SKIP_SAS   = {"SA012_10445982"}

# MS band file keywords — note 'red edge' has a space
MS_BAND_KEYWORDS = [
    "mosaic_red",       # Band 1 — MS Red
    "mosaic_green",     # Band 2 — MS Green
    "mosaic_nir",       # Band 3 — NIR
    "mosaic_red edge",  # Band 4 — Red-edge (space in filename)
]
MS_BAND_NAMES = ["MS_Red", "MS_Green", "NIR", "Red_Edge"]

print("Config ready.")
print("MS band keywords:")
for kw, name in zip(MS_BAND_KEYWORDS, MS_BAND_NAMES):
    print(f"  '{kw}' -> {name}")

Config ready.
MS band keywords:
  'mosaic_red' -> MS_Red
  'mosaic_green' -> MS_Green
  'mosaic_nir' -> NIR
  'mosaic_red edge' -> Red_Edge


## 1. Helper functions

In [2]:
def get_sa_id(folder_name):
    return folder_name.split("_", 1)[-1]


def find_ms_band(sa_dir, keyword):
    """
    Find a single-band MS mosaic file in sa_dir matching keyword.
    Handles filenames with spaces (e.g. 'mosaic_red edge').
    Returns Path or None.
    """
    for f in sa_dir.iterdir():
        if keyword in f.name and f.suffix in (".tif", ".tiff"):
            # Exclude the stacked file itself
            if "stack_MS4" not in f.name:
                return f
    return None


def read_aligned(path, ref_height, ref_width, ref_transform, ref_crs):
    """
    Read a raster band and resample to reference grid if dimensions differ.
    Returns numpy array of shape (ref_height, ref_width).
    """
    with rasterio.open(path) as src:
        if src.height == ref_height and src.width == ref_width:
            return src.read(1).astype(np.uint16)
        # Resample to reference grid
        data = np.empty((ref_height, ref_width), dtype=np.float32)
        rasterio.warp.reproject(
            source=rasterio.band(src, 1),
            destination=data,
            src_transform=src.transform,
            src_crs=src.crs,
            dst_transform=ref_transform,
            dst_crs=ref_crs,
            resampling=Resampling.bilinear,
        )
        return data.astype(np.uint16)


def pad_array(arr, target_h, target_w, pad_value=0):
    _, h, w = arr.shape
    if h == target_h and w == target_w:
        return arr
    out = np.full((arr.shape[0], target_h, target_w),
                  fill_value=pad_value, dtype=arr.dtype)
    out[:, :h, :w] = arr
    return out


def read_window(src, row_off, col_off, tile_h, tile_w, pad_value=0):
    h, w   = src.height, src.width
    read_h = min(tile_h, h - row_off)
    read_w = min(tile_w, w - col_off)
    data   = src.read(window=Window(col_off, row_off, read_w, read_h))
    return pad_array(data, tile_h, tile_w, pad_value=pad_value)


def tile_transform(base_tf, col_off, row_off):
    x = base_tf.c + col_off * base_tf.a
    y = base_tf.f + row_off * base_tf.e
    return rasterio.Affine(base_tf.a, 0.0, x, 0.0, base_tf.e, y)


def save_tile(arr, path, transform, crs, dtype="uint16", nodata=None):
    with rasterio.open(
        path, "w", driver="GTiff", dtype=dtype,
        width=arr.shape[2], height=arr.shape[1], count=arr.shape[0],
        crs=crs, transform=transform, compress="lzw", nodata=nodata,
    ) as dst:
        dst.write(arr)


print("Helpers defined.")

Helpers defined.


## 2. Check MS band availability across all SAs


In [3]:
sa_folders = sorted([
    d for d in SA_ROOT.iterdir()
    if d.is_dir() and d.name.startswith("SA") and d.name not in SKIP_SAS
])

print(f"Checking {len(sa_folders)} SAs for MS band availability ...\n")
print(f"{'SA':30s}  {'Red':>5}  {'Green':>5}  {'NIR':>5}  {'RedEdge':>7}  {'Status'}")
print("-" * 70)

all_ok = True
for sa_dir in sa_folders:
    sa_id   = get_sa_id(sa_dir.name)
    found   = []
    missing = []
    for kw, name in zip(MS_BAND_KEYWORDS, MS_BAND_NAMES):
        f = find_ms_band(sa_dir, kw)
        if f is not None:
            found.append("OK")
        else:
            found.append("MISS")
            missing.append(name)
            all_ok = False
    status = "OK" if not missing else f"MISSING: {missing}"
    print(f"  {sa_dir.name:30s}  {found[0]:>5}  {found[1]:>5}  "
          f"{found[2]:>5}  {found[3]:>7}  {status}")

print()
if all_ok:
    print("All 4 MS bands found for all SAs. Ready to build stacks.")
else:
    print("WARNING — some bands are missing. Check the paths above before proceeding.")

Checking 19 SAs for MS band availability ...

SA                                Red  Green    NIR  RedEdge  Status
----------------------------------------------------------------------
  SA010_10305790                     OK     OK     OK       OK  OK
  SA011_10445787                     OK     OK     OK       OK  OK
  SA013_10565691                     OK     OK     OK       OK  OK
  SA014_10595716                     OK     OK     OK       OK  OK
  SA015_10605304                     OK     OK     OK       OK  OK
  SA016_10765685                     OK     OK     OK       OK  OK
  SA017_10865406                     OK     OK     OK       OK  OK
  SA018_10865805                     OK     OK     OK       OK  OK
  SA019_11125706                     OK     OK     OK       OK  OK
  SA01_9545448                       OK     OK     OK       OK  OK
  SA020_11226291                     OK     OK     OK       OK  OK
  SA02_9805753                       OK     OK     OK       OK  OK
  SA03_996

## 3. Build MS 4-band stack per SA

Output: `{sa_id}_stack_MS4.tif` saved inside each SA folder.


In [4]:
print(f"Building 4-band MS stacks for {len(sa_folders)} SAs ...\n")
build_results = []

for sa_dir in sa_folders:
    sa_name  = sa_dir.name
    sa_id    = get_sa_id(sa_name)
    out_path = sa_dir / f"{sa_id}_stack_MS4.tif"

    if out_path.exists():
        print(f"  [SKIP] {sa_name} — stack already exists", flush=True)
        build_results.append({"SA": sa_name, "status": "skipped"})
        continue

    # Locate all 4 band files
    band_paths = [find_ms_band(sa_dir, kw) for kw in MS_BAND_KEYWORDS]
    missing = [MS_BAND_NAMES[i] for i, p in enumerate(band_paths) if p is None]
    if missing:
        print(f"  [WARN] {sa_name} — missing bands: {missing}, skipping", flush=True)
        build_results.append({"SA": sa_name, "status": f"missing {missing}"})
        continue

    # Use MS Red as reference grid
    with rasterio.open(band_paths[0]) as ref:
        meta      = ref.meta.copy()
        transform = ref.transform
        crs       = ref.crs
        height    = ref.height
        width     = ref.width
        band1     = ref.read(1).astype(np.uint16)

    # Read remaining bands aligned to reference
    band2 = read_aligned(band_paths[1], height, width, transform, crs)
    band3 = read_aligned(band_paths[2], height, width, transform, crs)
    band4 = read_aligned(band_paths[3], height, width, transform, crs)

    # Write 4-band stack
    out_meta = meta.copy()
    out_meta.update({
        "count":   4,
        "dtype":   "uint16",
        "compress": "lzw",
    })

    with rasterio.open(out_path, "w", **out_meta) as dst:
        dst.write(band1, 1)
        dst.write(band2, 2)
        dst.write(band3, 3)
        dst.write(band4, 4)
        dst.update_tags(
            Band1="MS_Red",
            Band2="MS_Green",
            Band3="NIR",
            Band4="Red_Edge",
        )

    print(f"  [OK]   {sa_name} -> {out_path.name}  "
          f"({height}x{width} px)", flush=True)
    build_results.append({"SA": sa_name, "status": "ok"})

print("\nStack building complete.")
display(pd.DataFrame(build_results))

Building 4-band MS stacks for 19 SAs ...

  [OK]   SA010_10305790 -> 10305790_stack_MS4.tif  (10900x11125 px)
  [OK]   SA011_10445787 -> 10445787_stack_MS4.tif  (11490x11186 px)
  [OK]   SA013_10565691 -> 10565691_stack_MS4.tif  (11460x11517 px)
  [OK]   SA014_10595716 -> 10595716_stack_MS4.tif  (11591x11473 px)
  [OK]   SA015_10605304 -> 10605304_stack_MS4.tif  (11930x11585 px)
  [OK]   SA016_10765685 -> 10765685_stack_MS4.tif  (11750x11810 px)
  [OK]   SA017_10865406 -> 10865406_stack_MS4.tif  (11420x11173 px)
  [OK]   SA018_10865805 -> 10865805_stack_MS4.tif  (11263x11772 px)
  [OK]   SA019_11125706 -> 11125706_stack_MS4.tif  (12155x11678 px)
  [OK]   SA01_9545448 -> 9545448_stack_MS4.tif  (11873x11972 px)
  [OK]   SA020_11226291 -> 11226291_stack_MS4.tif  (11083x11249 px)
  [OK]   SA02_9805753 -> 9805753_stack_MS4.tif  (11559x11289 px)
  [OK]   SA03_9965805 -> 9965805_stack_MS4.tif  (11631x11350 px)
  [OK]   SA04_10085703 -> 10085703_stack_MS4.tif  (11395x11693 px)
  [OK]   SA05_10

,SA,status
0,SA010_10305790,ok
1,SA011_10445787,ok
2,SA013_10565691,ok
3,SA014_10595716,ok
4,SA015_10605304,ok
5,SA016_10765685,ok
6,SA017_10865406,ok
7,SA018_10865805,ok
8,SA019_11125706,ok
9,SA01_9545448,ok


## 4. Verify stacks — check band stats for first 3 SAs

Confirms all 4 bands have reasonable value ranges and no collapsed bands.

In [5]:
print("Verifying MS4 stacks (first 3 SAs) ...\n")
print(f"{'SA':20s}  {'Band':>10}  {'Min':>8}  {'Max':>8}  "
      f"{'Mean':>8}  {'Std':>8}  {'Zero%':>6}")
print("-" * 75)

checked = 0
for sa_dir in sa_folders:
    if checked >= 3:
        break
    sa_id    = get_sa_id(sa_dir.name)
    ms4_path = sa_dir / f"{sa_id}_stack_MS4.tif"
    if not ms4_path.exists():
        continue

    with rasterio.open(ms4_path) as src:
        for b in range(1, 5):
            data  = src.read(b).astype(np.float32)
            valid = data[data > 0]
            zero_pct = (data == 0).mean() * 100
            print(
                f"  {sa_dir.name:20s}  "
                f"{MS_BAND_NAMES[b-1]:>10}  "
                f"{valid.min() if len(valid)>0 else 0:>8.0f}  "
                f"{valid.max() if len(valid)>0 else 0:>8.0f}  "
                f"{valid.mean() if len(valid)>0 else 0:>8.0f}  "
                f"{valid.std()  if len(valid)>0 else 0:>8.0f}  "
                f"{zero_pct:>6.1f}%"
            )
    print()
    checked += 1

Verifying MS4 stacks (first 3 SAs) ...

SA                          Band       Min       Max      Mean       Std   Zero%
---------------------------------------------------------------------------
  SA010_10305790            MS_Red      4028     65365     20081      7427     4.2%
  SA010_10305790          MS_Green      3899     65295     16059      5719     4.2%
  SA010_10305790               NIR      4148     55629     17375      5903     4.2%
  SA010_10305790          Red_Edge      4028     65365     20081      7427     4.2%

  SA011_10445787            MS_Red      4750     60912     20075      5602    14.1%
  SA011_10445787          MS_Green      4189     65182     15769      4399    14.1%
  SA011_10445787               NIR      4695     44335     17586      4625    14.1%
  SA011_10445787          Red_Edge      4750     60912     20075      5602    14.1%

  SA013_10565691            MS_Red      4176     52997     20391      5326     4.4%
  SA013_10565691          MS_Green      3650 

## 5. Tile MS stacks

512x512 tiles with 256px stride.
Uses the same label rasters from Notebook 1 as the reference grid.

In [6]:
# Clear any existing MS tiles first
existing = list(MS_IMG_DIR.glob("*.tif"))
for f in existing:
    f.unlink()
# Also clear split subfolders if they exist
for split in ["train", "val", "test"]:
    split_dir = MS_IMG_DIR / split
    if split_dir.exists():
        shutil.rmtree(split_dir)

print(f"Cleared {len(existing)} existing MS tiles.")
print(f"Starting MS tiling: {TILE_SIZE}px tiles, {STRIDE}px stride ...\n",
      flush=True)

summary = []

for sa_dir in sa_folders:
    sa_name    = sa_dir.name
    sa_id      = get_sa_id(sa_name)
    ms4_path   = sa_dir / f"{sa_id}_stack_MS4.tif"
    label_path = LABEL_DIR / f"{sa_id}_labels.tif"

    missing = []
    if not ms4_path.exists():   missing.append("MS4 stack")
    if not label_path.exists(): missing.append("label raster")
    if missing:
        print(f"  [SKIP] {sa_name} — missing: {missing}", flush=True)
        summary.append({"SA": sa_name, "tiles_saved": 0,
                         "status": f"missing {missing}"})
        continue

    saved = 0

    with (rasterio.open(ms4_path)   as ms_src,
          rasterio.open(label_path) as lbl_src):

        height, width = lbl_src.height, lbl_src.width
        crs       = lbl_src.crs
        transform = lbl_src.transform

        # If MS stack dimensions differ from label, resample on the fly
        ms_matches = (ms_src.height == height and ms_src.width == width)

        for r_idx, row in enumerate(range(0, height, STRIDE)):
            for c_idx, col in enumerate(range(0, width, STRIDE)):

                tile_name = f"{sa_id}_r{r_idx:04d}_c{c_idx:04d}.tif"
                tf        = tile_transform(transform, col, row)

                # Read label tile first — discard if all background
                lbl_tile = read_window(
                    lbl_src, row, col, TILE_SIZE, TILE_SIZE,
                    pad_value=BACKGROUND
                )
                if np.all(lbl_tile == BACKGROUND):
                    continue

                if ms_matches:
                    ms_tile = read_window(
                        ms_src, row, col, TILE_SIZE, TILE_SIZE, pad_value=0
                    )
                else:
                    # Resample full MS stack to label grid then tile
                    # (only triggered if resolutions differ)
                    read_h = min(TILE_SIZE, height - row)
                    read_w = min(TILE_SIZE, width  - col)
                    ms_tile_raw = np.zeros(
                        (4, TILE_SIZE, TILE_SIZE), dtype=np.uint16
                    )
                    for b in range(1, 5):
                        dst_band = np.zeros(
                            (TILE_SIZE, TILE_SIZE), dtype=np.float32
                        )
                        rasterio.warp.reproject(
                            source=rasterio.band(ms_src, b),
                            destination=dst_band,
                            src_transform=ms_src.transform,
                            src_crs=ms_src.crs,
                            dst_transform=tf,
                            dst_crs=crs,
                            resampling=Resampling.bilinear,
                        )
                        ms_tile_raw[b-1] = dst_band.astype(np.uint16)
                    ms_tile = ms_tile_raw

                save_tile(ms_tile, MS_IMG_DIR / tile_name,
                          tf, crs, dtype="uint16")
                saved += 1

    print(f"  {sa_name:30s}  {saved:5d} tiles", flush=True)
    summary.append({"SA": sa_name, "tiles_saved": saved, "status": "ok"})

summary_df = pd.DataFrame(summary)
total = summary_df["tiles_saved"].sum()
print(f"\nTotal MS tiles saved: {total:,}")
display(summary_df)

Cleared 0 existing MS tiles.
Starting MS tiling: 512px tiles, 256px stride ...

  SA010_10305790                   1856 tiles
  SA011_10445787                    688 tiles
  SA013_10565691                   2009 tiles
  SA014_10595716                   2008 tiles
  SA015_10605304                   2024 tiles
  SA016_10765685                   1237 tiles
  SA017_10865406                   1298 tiles
  SA018_10865805                   1165 tiles
  SA019_11125706                    836 tiles
  SA01_9545448                     1889 tiles
  SA020_11226291                     21 tiles
  SA02_9805753                      364 tiles
  SA03_9965805                      153 tiles
  SA04_10085703                     904 tiles
  SA05_10125706                    1887 tiles
  SA06_10145814                    1955 tiles
  SA07_10165835                     424 tiles
  SA08_10165859                     682 tiles
  SA09_10185850                     791 tiles

Total MS tiles saved: 22,191


,SA,tiles_saved,status
0,SA010_10305790,1856,ok
1,SA011_10445787,688,ok
2,SA013_10565691,2009,ok
3,SA014_10595716,2008,ok
4,SA015_10605304,2024,ok
5,SA016_10765685,1237,ok
6,SA017_10865406,1298,ok
7,SA018_10865805,1165,ok
8,SA019_11125706,836,ok
9,SA01_9545448,1889,ok


## 6. Verify MS tile count matches RGB and label tiles

In [7]:
rgb_flat = list((TILES_DIR / "rgb" / "images").glob("*.tif"))
ms_flat  = list(MS_IMG_DIR.glob("*.tif"))
lbl_flat = list((TILES_DIR / "labels").glob("*.tif"))

# Labels may already be in split subfolders — count recursively
lbl_all  = list((TILES_DIR / "labels").rglob("*.tif"))
rgb_all  = list((TILES_DIR / "rgb" / "images").rglob("*.tif"))

print(f"MS tiles (flat)    : {len(ms_flat):,}")
print(f"RGB tiles (all)    : {len(rgb_all):,}")
print(f"Label tiles (all)  : {len(lbl_all):,}")

if len(ms_flat) == len(rgb_all):
    print("\nMS tile count matches RGB. Ready to split.")
else:
    diff = len(rgb_all) - len(ms_flat)
    print(f"\nWARNING — MS has {abs(diff)} {'fewer' if diff>0 else 'more'} tiles than RGB.")
    print("This may be due to resolution differences between MS and label rasters.")
    print("Check the SA-level counts in the summary table above.")

MS tiles (flat)    : 22,191
RGB tiles (all)    : 22,191
Label tiles (all)  : 22,191

MS tile count matches RGB. Ready to split.


## 7. Move MS tiles into existing split folders


In [8]:
# Same SA assignment as Stage 3
ASSIGNMENT = {
    "11226291": "train",
    "9965805":  "train",
    "9805753":  "train",
    "10565691": "train",
    "10305790": "train",
    "10865406": "train",
    "10165835": "train",
    "10085703": "train",
    "10185850": "train",
    "9545448":  "train",
    "10445787": "train",
    "11125706": "train",
    "10165859": "train",
    "10145814": "val",
    "10595716": "val",
    "10765685": "val",
    "10125706": "test",
    "10605304": "test",
    "10865805": "test",
}

splits = ["train", "val", "test"]

# Create MS split subdirectories
for split in splits:
    (MS_IMG_DIR / split).mkdir(parents=True, exist_ok=True)

def get_sa_id_from_tile(tile_name):
    return tile_name.split("_r")[0]

tiles  = sorted(MS_IMG_DIR.glob("*.tif"))
moved  = 0
skipped = 0

print(f"Moving {len(tiles):,} MS tiles into split folders ...", flush=True)

for tile in tiles:
    sa_id = get_sa_id_from_tile(tile.name)
    split = ASSIGNMENT.get(sa_id)
    if split is None:
        print(f"  [WARN] No assignment for SA '{sa_id}'", flush=True)
        skipped += 1
        continue
    dest = MS_IMG_DIR / split / tile.name
    shutil.move(str(tile), str(dest))
    moved += 1

print(f"  Moved: {moved:,}  Skipped: {skipped}", flush=True)
print("\nDone.")

Moving 22,191 MS tiles into split folders ...
  Moved: 22,191  Skipped: 0

Done.


## 8. Final summary

In [9]:
rows = []
for split in splits:
    ms_tiles  = list((MS_IMG_DIR / split).glob("*.tif"))
    rgb_tiles = list((TILES_DIR / "rgb" / "images" / split).glob("*.tif"))
    lbl_tiles = list((TILES_DIR / "labels" / split).glob("*.tif"))
    sa_list   = sorted({get_sa_id_from_tile(t.name) for t in ms_tiles})
    match     = "OK" if len(ms_tiles) == len(rgb_tiles) == len(lbl_tiles) \
                else "MISMATCH"
    rows.append({
        "split":      split,
        "n_SAs":      len(sa_list),
        "MS_tiles":   len(ms_tiles),
        "RGB_tiles":  len(rgb_tiles),
        "Lbl_tiles":  len(lbl_tiles),
        "match":      match,
    })

final_df = pd.DataFrame(rows)
print("Final split summary:")
display(final_df)

all_match = all(r["match"] == "OK" for r in rows)
if all_match:
    print("\nAll splits match across MS, RGB and labels.")
    print("Ready to run Notebook 4 with MODALITY = 'ms'")
else:
    print("\nWARNING — mismatch detected. Check tile counts above.")

Final split summary:


,split,n_SAs,MS_tiles,RGB_tiles,Lbl_tiles,match
0,train,13,11915,11915,11915,OK
1,val,3,5200,5200,5200,OK
2,test,3,5076,5076,5076,OK



All splits match across MS, RGB and labels.
Ready to run Notebook 4 with MODALITY = 'ms'
